# Mapping Pipeline
This notebook will deploy the pipeline that performs the mapping from sensor data tables to timestamped RDF triples. We do this to easily incorporate the telemetry data with the model graph defined later.

In [0]:
%pip install --upgrade databricks-sdk>=0.61.0
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
%run /Users/ankit.yadav@databricks.com/frozen-potato-digital-twin/0-Parameters

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors.platform import ResourceConflict
from databricks.sdk.service.pipelines import (
    PipelineLibrary,
    FileLibrary,
    PipelinesEnvironment,
)

w = WorkspaceClient()
notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
cwd = notebook_path.rsplit("/", 1)[0]

pipeline_conf = dict(
    name=MAPPING_PIPELINE_NAME,
    catalog=TRIPLE_CATALOG,
    schema=TRIPLE_SCHEMA,
    development=True,
    serverless=True,
    environment=PipelinesEnvironment(
        dependencies=["git+https://github.com/aktungmak/spark-r2r.git"],
    ),
    libraries=[
        PipelineLibrary(
            file=FileLibrary(path=cwd + "/mapping_pipeline/src/dt_mapping.py")
        )
    ],
    configuration={
        "bundle.sourcePath": "./src",
        "triple_table": TRIPLE_TABLE,
        "bronze_table": BRONZE_TABLE,
    },
)

In [0]:
try:
    pipeline = w.pipelines.create(allow_duplicate_names=False, **pipeline_conf)
    print(f"Pipeline created with ID: {pipeline.pipeline_id}")
except ResourceConflict:
    pipeline = next(w.pipelines.list_pipelines(filter=f"name LIKE '{MAPPING_PIPELINE_NAME}'"))
    w.pipelines.update(pipeline_id=pipeline.pipeline_id, allow_duplicate_names=False, **pipeline_conf)
    print(f"Updated existing pipeline with ID: {pipeline.pipeline_id} updated")

Pipeline created with ID: 56512ccf-9072-4fcd-a9e1-a87159b271c8


In [0]:
import time

# Start the pipeline update
update = w.pipelines.start_update(pipeline_id=pipeline.pipeline_id)
print(f"Started pipeline update: {update.update_id}")

# Wait for completion
print("Waiting for pipeline to complete...")
while True:
    update_status = w.pipelines.get_update(pipeline_id=pipeline.pipeline_id, update_id=update.update_id)
    state = update_status.update.state.value
    
    print(f"Pipeline state: {state}")
    
    if state in ["COMPLETED", "FAILED", "CANCELED"]:
        if state == "COMPLETED":
            print("✅ Pipeline completed successfully!")
            break
        else:
            raise Exception(f"❌ Pipeline failed with state: {state}")
    
    time.sleep(30)  # Check every 30 seconds

Started pipeline update: 6abcf803-2537-4dab-8a89-af22a4cd316c
Waiting for pipeline to complete...
Pipeline state: CREATED
Pipeline state: INITIALIZING
Pipeline state: INITIALIZING
Pipeline state: COMPLETED
✅ Pipeline completed successfully!
